In [1]:
!pip install -q -U transformers accelerate sentencepiece safetensors
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.0 MB/s eta 0:00:00
True Tesla T4


In [2]:
!cp -r /kaggle/input/datasets/rifatbinreza/alta2026-pipeline/alta2026_pipeline /kaggle/working/
%cd /kaggle/working/alta2026_pipeline
!ls

/kaggle/working/alta2026_pipeline
answer_initial.csv	  infer.py	    thresholds.py
artifacts		  metadata	    train.csv
blend.py		  metrics.py	    transformer_multitask.py
classical_baseline.py	  README.md	    validate_answer.py
classical_baseline_v2.py  requirements.txt  valid.csv
evaluate.py		  run_baseline.sh


In [3]:
!python classical_baseline.py --train train.csv --valid valid.csv

{
  "component_scores": {
    "sentiment-en-AU": 0.8443665962598527,
    "sentiment-en-UK": 0.9090572515230049,
    "sarcasm-en-AU": 0.6402292095210109,
    "sarcasm-en-UK": 0.6395495742927767
  },
  "alta_score": 0.7419580852763147,
  "thresholds": {
    "sentiment-en-AU": 0.48,
    "sentiment-en-UK": 0.5,
    "sarcasm-en-AU": 0.13,
    "sarcasm-en-UK": 0.13,
    "alta_score": 0.7419580852763147
  }
}


In [4]:
%%writefile /kaggle/working/alta2026_pipeline/transformer_multitask.py
from __future__ import annotations

import argparse
import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

from metrics import alta_score, competition_stratify_key
from thresholds import optimize_thresholds, apply_thresholds, save_thresholds

try:
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except ImportError as e:
    raise SystemExit(
        "transformers is required. Install with: pip install -r requirements.txt"
    ) from e


@dataclass
class Config:
    model_name: str = "microsoft/deberta-v3-large"
    max_length: int = 384
    batch_size: int = 8
    epochs: int = 4
    lr: float = 1.5e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.15
    sentiment_loss_weight: float = 1.0
    sarcasm_loss_weight: float = 1.25
    sarcasm_pos_weight: float = 1.0
    gradient_accumulation_steps: int = 4
    fp16: bool = False
    seed: int = 42
    n_folds: int = 5
    early_stopping_patience: int = 2


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


SPECIAL_TOKENS = ["[SRC_GOOGLE]", "[SRC_REDDIT]", "[VAR_EN_AU]", "[VAR_EN_UK]"]

def add_prefix(df: pd.DataFrame) -> pd.Series:
    return (
        "[SRC_" + df.source.str.upper() + "] [VAR_" + df.variety.str.replace("-", "_").str.upper() + "] "
        + df.text.fillna("")
    )


class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.texts = add_prefix(self.df).tolist()
        self.max_length = max_length
        self.with_labels = with_labels

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i], max_length=self.max_length,
            truncation=True, padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.with_labels:
            item["sentiment"] = torch.tensor(int(self.df.loc[i, "sentiment"]), dtype=torch.long)
            item["sarcasm"] = torch.tensor(int(self.df.loc[i, "sarcasm"]), dtype=torch.long)
        return item


class MultiTaskModel(nn.Module):
    def __init__(self, name: str, dropout: float):
        super().__init__()
        # Force float32: some hub checkpoints default-load in fp16, which
        # then mismatches with our plain-float32 classifier heads.
        self.encoder = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        h = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Linear(h, 2)
        self.sarc_head = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out = self.encoder(**kwargs)
        pooled = out.last_hidden_state[:, 0]
        pooled = self.drop(pooled)
        return self.sent_head(pooled), self.sarc_head(pooled)


def evaluate(model, loader, device):
    model.eval(); ps, pz = [], []; ys, yz = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            ps.append(torch.softmax(a, -1)[:, 1].cpu().numpy())
            pz.append(torch.softmax(b, -1)[:, 1].cpu().numpy())
            ys.append(batch["sentiment"].numpy()); yz.append(batch["sarcasm"].numpy())
    return np.concatenate(ps), np.concatenate(pz), np.concatenate(ys), np.concatenate(yz)


def train_one_fold(train_df, val_df, cfg, out_dir, fold):
    set_seed(cfg.seed + fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    model = MultiTaskModel(cfg.model_name, cfg.dropout)
    model.encoder.resize_token_embeddings(len(tokenizer))
    model = model.to(device)

    tr_ds = TextDataset(train_df, tokenizer, cfg.max_length, True)
    va_ds = TextDataset(val_df, tokenizer, cfg.max_length, True)
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_loader) * cfg.epochs // cfg.gradient_accumulation_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(steps * cfg.warmup_ratio), steps
    )

    s_loss = nn.CrossEntropyLoss()
    z_weight = torch.tensor([1.0, cfg.sarcasm_pos_weight], device=device)
    z_loss = nn.CrossEntropyLoss(weight=z_weight)
    best = -1.0
    best_state = None
    epochs_since_improve = 0

    for epoch in range(cfg.epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            loss = (cfg.sentiment_loss_weight * s_loss(a, batch["sentiment"].to(device))
                    + cfg.sarcasm_loss_weight * z_loss(b, batch["sarcasm"].to(device)))
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        ps, pz, ys, yz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        scores, comp = alta_score(val_df, tmp)
        print(f"fold={fold} epoch={epoch+1} score={comp:.5f} {scores}")
        if comp > best:
            best = comp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= cfg.early_stopping_patience:
                print(f"fold={fold}: no improvement for {cfg.early_stopping_patience} epochs, stopping early at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), out_dir / f"fold{fold}.pt")
    meta = {"fold": fold, "best_validation_score": best, "config": asdict(cfg)}
    with open(out_dir / f"fold{fold}.json", "w") as f: json.dump(meta, f, indent=2)
    return model, tokenizer, device


def cv_train(train_csv: str, out_dir: str, cfg: Config, external_valid_csv: str | None = None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(train_csv)
    keys = competition_stratify_key(df)
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)
    oof = df[["source", "variety", "text", "sentiment", "sarcasm"]].copy()
    oof["p_sentiment"] = np.nan; oof["p_sarcasm"] = np.nan

    ext = pd.read_csv(external_valid_csv) if external_valid_csv else None
    ext_ps_all, ext_pz_all = [], []

    for fold, (tr, va) in enumerate(skf.split(df, keys)):
        model, tokenizer, device = train_one_fold(df.iloc[tr].copy(), df.iloc[va].copy(), cfg, out, fold)
        va_ds = TextDataset(df.iloc[va].copy(), tokenizer, cfg.max_length, True)
        va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)
        ps, pz, _, _ = evaluate(model, va_loader, device)
        oof.loc[df.index[va], "p_sentiment"] = ps
        oof.loc[df.index[va], "p_sarcasm"] = pz

        if ext is not None:
            ext_ds = TextDataset(ext.copy(), tokenizer, cfg.max_length, False)
            ext_loader = DataLoader(ext_ds, batch_size=cfg.batch_size * 2, shuffle=False)
            model.eval(); eps=[]; epz=[]
            with torch.no_grad():
                for batch in ext_loader:
                    ids=batch["input_ids"].to(device); mask=batch["attention_mask"].to(device)
                    tt=batch.get("token_type_ids")
                    if tt is not None: tt=tt.to(device)
                    a,b=model(ids,mask,tt)
                    eps.append(torch.softmax(a,-1)[:,1].cpu().numpy())
                    epz.append(torch.softmax(b,-1)[:,1].cpu().numpy())
            ext_ps_all.append(np.concatenate(eps)); ext_pz_all.append(np.concatenate(epz))

    oof.to_csv(out / "oof_probabilities.csv", index=False)
    pred = oof[["source", "variety", "text"]].copy()
    pred["sentiment"] = (oof.p_sentiment >= .5).astype(int)
    pred["sarcasm"] = (oof.p_sarcasm >= .5).astype(int)
    scores, comp = alta_score(df, pred)
    print("OOF @0.5", scores, comp)

    if ext is not None:
        ext_probs = ext[["source", "variety", "text"]].copy()
        ext_probs["p_sentiment"] = np.mean(ext_ps_all, axis=0)
        ext_probs["p_sarcasm"] = np.mean(ext_pz_all, axis=0)
        ext_probs.to_csv(out / "valid_probabilities.csv", index=False)
        ext_pred = ext_probs[["source", "variety", "text"]].copy()
        ext_pred["sentiment"] = (ext_probs.p_sentiment >= .5).astype(int)
        ext_pred["sarcasm"] = (ext_probs.p_sarcasm >= .5).astype(int)
        ext_scores, ext_comp = alta_score(ext, ext_pred)
        print("External valid @0.5", ext_scores, ext_comp)

def calibrate(valid_csv: str, probs_csv: str, out_dir: str):
    valid = pd.read_csv(valid_csv)
    probs = pd.read_csv(probs_csv)
    th = optimize_thresholds(valid, probs)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    save_thresholds(th, os.path.join(out_dir, "thresholds.json"))
    pred = apply_thresholds(probs, th)
    scores, comp = alta_score(valid, pred)
    pred.to_csv(os.path.join(out_dir, "calibrated_valid_predictions.csv"), index=False)
    print(json.dumps({"scores": scores, "alta_score": comp, "thresholds": th}, indent=2))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["cv"], default="cv")
    ap.add_argument("--train", default="../train.csv")
    ap.add_argument("--valid", default=None)
    ap.add_argument("--out", default="artifacts/transformer")
    ap.add_argument("--model", default="microsoft/deberta-v3-large")
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--max-length", type=int, default=384)
    ap.add_argument("--n-folds", type=int, default=5)
    ap.add_argument("--patience", type=int, default=2, help="stop a fold early after this many epochs with no score improvement")
    ap.add_argument("--grad-accum", type=int, default=4, help="gradient accumulation steps; raise this and lower --batch-size together if you hit CUDA out-of-memory")
    args, _unknown = ap.parse_known_args()
    cfg = Config(model_name=args.model, epochs=args.epochs, batch_size=args.batch_size,
                 max_length=args.max_length, n_folds=args.n_folds, early_stopping_patience=args.patience,
                 gradient_accumulation_steps=args.grad_accum)
    cv_train(args.train, args.out, cfg, args.valid)

Overwriting /kaggle/working/alta2026_pipeline/transformer_multitask.py


In [5]:
!python transformer_multitask.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/transformer_base \
  --model microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 32 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
config.json: 100%|█████████████████████████████| 579/579 [00:00<00:00, 3.30MB/s]
tokenizer_config.json: 100%|██████████████████| 52.0/52.0 [00:00<00:00, 392kB/s]
spm.model: 100%|███████████████████████████| 2.46M/2.46M [00:00<00:00, 3.07MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin: 100%|██████████████████████| 371M/371M [00:03<00:00, 115MB/s]
Loading weights: 100%|███████████████████████| 198/198 [00:00<00:00, 812.35it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.b

In [6]:
from thresholds import optimize_thresholds, save_thresholds, apply_thresholds
from metrics import alta_score
import pandas as pd

valid = pd.read_csv('valid.csv')
probs = pd.read_csv('artifacts/transformer_base/valid_probabilities.csv')
th = optimize_thresholds(valid, probs)
save_thresholds(th, 'artifacts/transformer_base/thresholds.json')
pred = apply_thresholds(probs, th)
scores, final = alta_score(valid, pred)
print("BASE MODEL CALIBRATED:", scores, final)

BASE MODEL CALIBRATED: {'sentiment-en-AU': 0.9215043957246563, 'sentiment-en-UK': 0.9506815415548697, 'sarcasm-en-AU': 0.7654048340364488, 'sarcasm-en-UK': 0.6927902621722847} 0.8071473289484705


In [7]:
!python transformer_multitask.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/transformer_large \
  --model microsoft/deberta-v3-large \
  --epochs 8 \
  --batch-size 4 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 8

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
config.json: 100%|█████████████████████████████| 580/580 [00:00<00:00, 2.10MB/s]
tokenizer_config.json: 100%|██████████████████| 52.0/52.0 [00:00<00:00, 293kB/s]
spm.model: 100%|███████████████████████████| 2.46M/2.46M [00:00<00:00, 3.93MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
pytorch_model.bin: 100%|██████████████████████| 874M/874M [00:05<00:00, 150MB/s]
Loading weights: 100%|███████████████████████| 390/390 [00:00<00:00, 761.91it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense

In [8]:
valid = pd.read_csv('valid.csv')
probs = pd.read_csv('artifacts/transformer_large/valid_probabilities.csv')
th = optimize_thresholds(valid, probs)
save_thresholds(th, 'artifacts/transformer_large/thresholds.json')
pred = apply_thresholds(probs, th)
scores, final = alta_score(valid, pred)
print("LARGE MODEL CALIBRATED:", scores, final)

LARGE MODEL CALIBRATED: {'sentiment-en-AU': 0.9321710386789623, 'sentiment-en-UK': 0.9636123680241327, 'sarcasm-en-AU': 0.7891142690286159, 'sarcasm-en-UK': 0.7821616979511716} 0.857166368315067


In [9]:
%%writefile sweep_blend.py
import sys
import pandas as pd
from thresholds import optimize_thresholds, apply_thresholds
from metrics import alta_score

prob_b_path = sys.argv[1] if len(sys.argv) > 1 else "artifacts/transformer_large/valid_probabilities.csv"

valid = pd.read_csv("valid.csv")
a = pd.read_csv("artifacts/classical/valid_probabilities.csv")
b = pd.read_csv(prob_b_path)

cols = ["source", "variety", "text"]
assert a[cols].equals(b[cols]) and a[cols].equals(valid[cols]), "row mismatch"

results = []
for wa in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    wb = 1 - wa
    p = valid[cols].copy()
    p["p_sentiment"] = wa * a["p_sentiment"] + wb * b["p_sentiment"]
    p["p_sarcasm"] = wa * a["p_sarcasm"] + wb * b["p_sarcasm"]
    th = optimize_thresholds(valid, p)
    pred = apply_thresholds(p, th)
    scores, final = alta_score(valid, pred)
    results.append({"weight_classical": wa, "weight_transformer": wb, "alta_score": final, **scores})
    print(f"wa={wa:.1f} wb={wb:.1f} -> ALTA={final:.4f}  {scores}")

df = pd.DataFrame(results).sort_values("alta_score", ascending=False)
print(f"\nBest blend for {prob_b_path}:")
print(df.iloc[0])

Writing sweep_blend.py


In [10]:
!python sweep_blend.py artifacts/transformer_large/valid_probabilities.csv

wa=0.0 wb=1.0 -> ALTA=0.8572  {'sentiment-en-AU': 0.9321710386789623, 'sentiment-en-UK': 0.9636123680241327, 'sarcasm-en-AU': 0.7891142690286159, 'sarcasm-en-UK': 0.7821616979511716}
wa=0.1 wb=0.9 -> ALTA=0.8567  {'sentiment-en-AU': 0.9322819698463112, 'sentiment-en-UK': 0.9636123680241327, 'sarcasm-en-AU': 0.7814128007938477, 'sarcasm-en-UK': 0.7810798548094373}
wa=0.2 wb=0.8 -> ALTA=0.8553  {'sentiment-en-AU': 0.929599462836457, 'sentiment-en-UK': 0.961024536367002, 'sarcasm-en-AU': 0.7833365777691259, 'sarcasm-en-UK': 0.7810798548094373}
wa=0.3 wb=0.7 -> ALTA=0.8552  {'sentiment-en-AU': 0.9296938775510204, 'sentiment-en-UK': 0.9610455141790291, 'sarcasm-en-AU': 0.7845731324544885, 'sarcasm-en-UK': 0.7806259431281478}
wa=0.4 wb=0.6 -> ALTA=0.8496  {'sentiment-en-AU': 0.9241264461844104, 'sentiment-en-UK': 0.9532423483808448, 'sarcasm-en-AU': 0.7787507846829881, 'sarcasm-en-UK': 0.7750899117960257}
wa=0.5 wb=0.5 -> ALTA=0.8438  {'sentiment-en-AU': 0.9184113300492611, 'sentiment-en-UK'

In [11]:
!python sweep_blend.py artifacts/transformer_base/valid_probabilities.csv

wa=0.0 wb=1.0 -> ALTA=0.8071  {'sentiment-en-AU': 0.9215043957246563, 'sentiment-en-UK': 0.9506815415548697, 'sarcasm-en-AU': 0.7654048340364488, 'sarcasm-en-UK': 0.6927902621722847}
wa=0.1 wb=0.9 -> ALTA=0.8021  {'sentiment-en-AU': 0.9215043957246563, 'sentiment-en-UK': 0.948073611708997, 'sarcasm-en-AU': 0.768125, 'sarcasm-en-UK': 0.6827600678475877}
wa=0.2 wb=0.8 -> ALTA=0.7995  {'sentiment-en-AU': 0.9241840369007999, 'sentiment-en-UK': 0.9506576512934369, 'sarcasm-en-AU': 0.7417034114643769, 'sarcasm-en-UK': 0.6747212260175985}
wa=0.3 wb=0.7 -> ALTA=0.7999  {'sentiment-en-AU': 0.9215043957246563, 'sentiment-en-UK': 0.948073611708997, 'sarcasm-en-AU': 0.7279978885011291, 'sarcasm-en-UK': 0.6783333333333333}
wa=0.4 wb=0.6 -> ALTA=0.7931  {'sentiment-en-AU': 0.9184113300492611, 'sentiment-en-UK': 0.9506815415548697, 'sarcasm-en-AU': 0.7222154767686579, 'sarcasm-en-UK': 0.6677187948350072}
wa=0.5 wb=0.5 -> ALTA=0.7911  {'sentiment-en-AU': 0.9156478052000441, 'sentiment-en-UK': 0.953266

In [12]:
!cd /kaggle/working/alta2026_pipeline && zip -r all_artifacts.zip artifacts/

  adding: artifacts/ (stored 0%)
  adding: artifacts/classical_rerun/ (stored 0%)
  adding: artifacts/classical_rerun/thresholds.json (deflated 41%)
  adding: artifacts/classical_rerun/valid_probabilities.csv (deflated 58%)
  adding: artifacts/classical_rerun/valid_predictions.csv (deflated 61%)
  adding: artifacts/classical/ (stored 0%)
  adding: artifacts/classical/thresholds.json (deflated 41%)
  adding: artifacts/classical/valid_probabilities.csv (deflated 58%)
  adding: artifacts/classical/valid_predictions.csv (deflated 61%)
  adding: artifacts/transformer_large/ (stored 0%)
  adding: artifacts/transformer_large/oof_probabilities.csv (deflated 59%)
  adding: artifacts/transformer_large/fold1.json (deflated 46%)
  adding: artifacts/transformer_large/fold0.pt (deflated 15%)
  adding: artifacts/transformer_large/thresholds.json (deflated 41%)
  adding: artifacts/transformer_large/fold0.json (deflated 46%)
  adding: artifacts/transformer_large/valid_probabilities.csv (deflated 59%)
 

In [13]:
!pip install -q -U transformers accelerate sentencepiece safetensors

In [14]:
!cp -r /kaggle/input/datasets/rifatbinreza/alta2026-pipeline/alta2026_pipeline /kaggle/working/
%cd /kaggle/working/alta2026_pipeline
!ls

/kaggle/working/alta2026_pipeline
all_artifacts.zip	  infer.py	    sweep_blend.py
answer_initial.csv	  metadata	    thresholds.py
artifacts		  metrics.py	    train.csv
blend.py		  __pycache__	    transformer_multitask.py
classical_baseline.py	  README.md	    validate_answer.py
classical_baseline_v2.py  requirements.txt  valid.csv
evaluate.py		  run_baseline.sh


In [15]:
import pandas as pd
from thresholds import apply_thresholds
import json

valid = pd.read_csv('valid.csv')
probs = pd.read_csv('/kaggle/input/datasets/rifatbinreza/result/valid_probabilities.csv')
th = json.load(open('/kaggle/input/datasets/rifatbinreza/result/thresholds.json'))

pred = apply_thresholds(probs, th)
pred.to_csv('answer.csv', index=False)
print(pred.columns.tolist(), len(pred))

['source', 'variety', 'text', 'sentiment', 'sarcasm'] 757


In [16]:
!python validate_answer.py --gold valid.csv --pred answer.csv
!zip answer.zip answer.csv
!unzip -l answer.zip

{'sentiment-en-AU': 0.9321710386789623, 'sentiment-en-UK': 0.9636123680241327, 'sarcasm-en-AU': 0.7891142690286159, 'sarcasm-en-UK': 0.7821616979511716}
ALTA score: 0.857166368315067
  adding: answer.csv (deflated 61%)
Archive:  answer.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   255693  2026-09-03 08:05   answer.csv
---------                     -------
   255693                     1 file


In [17]:
 !pwd

/kaggle/working/alta2026_pipeline


In [18]:
%%writefile idan_model.py
"""
idan_model.py — Incongruity-Aware Dual-Attention Network (IDAN)

Novel architecture (not a fine-tuned-transformer-only baseline):

  1. LITERAL BRANCH: multi-kernel 1D CNN over token embeddings, capturing
     local/surface-level sentiment cues (what the sentence "appears" to say).
  2. CONTEXTUAL BRANCH: pretrained transformer encoder (DeBERTa), capturing
     the sentence's actual discourse-level/contextual meaning.
  3. INCONGRUITY FUSION: the two branches are combined into a joint
     [literal ; contextual] representation, per token, then passed through
     a CBAM-style (Woo et al. 2018) dual attention module:
       - Channel attention: which FEATURE DIMENSIONS (of literal vs
         contextual signal) matter for this example.
       - Spatial attention: which TOKEN POSITIONS carry the strongest
         mismatch signal between literal and contextual meaning.
     We additionally compute an explicit incongruity vector
     (elementwise difference and product between literal- and
     contextual-branch pooled representations), following the standard
     NLI-style mismatch-feature trick, giving the model a direct numeric
     signal for "these two views of the sentence disagree."
  4. Fused representation -> two classification heads (sentiment, sarcasm),
     same multi-task setup as transformer_multitask.py, same source/variety
     control-token conditioning.

This is intentionally NOT a from-scratch pretrained language model -- that
would require far more data/compute than is available here and would
likely underperform a fine-tuned encoder. The novelty is the attention/
fusion mechanism built on top of existing pretrained representations,
consistent with how recent incongruity-based sarcasm papers (ACL/COLING
2025) structure their contributions.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class LiteralCNNBranch(nn.Module):
    """Multi-kernel 1D CNN over token embeddings -> per-token local feature map.
    Uses 'same' padding so output sequence length matches input length,
    which lets us align this branch token-for-token with the transformer's
    hidden states downstream.
    """
    def __init__(self, embed_dim: int, out_channels: int = 256, kernel_sizes=(2, 3, 4, 5)):
        super().__init__()
        per_k = out_channels // len(kernel_sizes)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, per_k, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.out_channels = per_k * len(kernel_sizes)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(self.out_channels)

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        x = token_embeds.transpose(1, 2)
        feats = []
        for conv in self.convs:
            f = self.act(conv(x))
            f = f[:, :, :token_embeds.size(1)]
            feats.append(f)
        out = torch.cat(feats, dim=1)
        out = out.transpose(1, 2)
        return self.norm(out)


class ChannelAttention(nn.Module):
    """CBAM-style channel attention: squeeze spatial (sequence) dimension via
    both avg- and max-pooling, pass through a shared MLP, sum, sigmoid.
    Tells the model which FEATURE CHANNELS (literal-branch dims vs
    contextual-branch dims) matter most for this example.

    reduction is raised to 16 (from the original CBAM paper's 16, not 8) and
    dropout is added inside the MLP -- on a dataset this small (~1.8k
    examples/fold), the extra squeeze-excite parameters are a real
    overfitting risk, so keep this module as lightweight as possible."""
    def __init__(self, channels: int, reduction: int = 16, dropout: float = 0.1):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, channels),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        avg_pool = (x * m).sum(dim=1) / denom
        max_pool = (x.masked_fill(m == 0, float("-inf"))).max(dim=1).values
        channel_att = torch.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool))
        return x * channel_att.unsqueeze(1)


class SpatialAttention(nn.Module):
    """CBAM-style spatial attention: squeeze channel dimension via avg- and
    max-pooling, concat, 1D conv over the sequence, sigmoid.
    Tells the model which TOKEN POSITIONS carry the strongest
    literal-vs-contextual mismatch (the actual incongruity signal)."""
    def __init__(self, kernel_size: int = 5):
        super().__init__()
        self.conv = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        avg_pool = x.mean(dim=2, keepdim=True)
        max_pool = x.max(dim=2, keepdim=True).values
        pooled = torch.cat([avg_pool, max_pool], dim=2).transpose(1, 2)
        att = torch.sigmoid(self.conv(pooled)).transpose(1, 2)
        att = att * mask.unsqueeze(-1)
        return x * att


class IncongruityDualAttention(nn.Module):
    """Fuses literal + contextual token representations, applies channel
    then spatial attention (CBAM ordering), and computes an explicit
    incongruity vector from the pooled branch representations."""
    def __init__(self, literal_dim: int, contextual_dim: int, proj_dim: int = 384):
        super().__init__()
        self.lit_proj = nn.Linear(literal_dim, proj_dim)
        self.ctx_proj = nn.Linear(contextual_dim, proj_dim)
        joint_dim = proj_dim * 2
        self.channel_att = ChannelAttention(joint_dim)
        self.spatial_att = SpatialAttention()
        self.fuse_norm = nn.LayerNorm(joint_dim)
        self.proj_dim = proj_dim

    def forward(self, literal_feats, contextual_feats, mask):
        lit = self.lit_proj(literal_feats)
        ctx = self.ctx_proj(contextual_feats)
        joint = torch.cat([lit, ctx], dim=-1)

        joint = self.channel_att(joint, mask)
        joint = self.spatial_att(joint, mask)
        joint = self.fuse_norm(joint)

        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        pooled_fused = (joint * m).sum(dim=1) / denom

        lit_pooled = (lit * m).sum(dim=1) / denom
        ctx_pooled = (ctx * m).sum(dim=1) / denom
        diff = lit_pooled - ctx_pooled
        prod = lit_pooled * ctx_pooled

        return torch.cat([pooled_fused, diff, prod], dim=-1)


class IDAN(nn.Module):
    """Full model: transformer encoder (contextual branch) + CNN (literal
    branch) + incongruity dual-attention fusion + two task heads."""
    def __init__(self, encoder, hidden_size: int, cnn_out_channels: int = 256,
                 proj_dim: int = 384, dropout: float = 0.15):
        super().__init__()
        self.encoder = encoder
        self.embed_layer = encoder.get_input_embeddings()
        self.literal_branch = LiteralCNNBranch(
            embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
        )
        self.fusion = IncongruityDualAttention(
            literal_dim=self.literal_branch.out_channels,
            contextual_dim=hidden_size,
            proj_dim=proj_dim,
        )
        fused_dim = proj_dim * 4
        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))
        self.sarc_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        enc_out = self.encoder(**kwargs)
        contextual_feats = enc_out.last_hidden_state

        token_embeds = self.embed_layer(input_ids)
        literal_feats = self.literal_branch(token_embeds)

        mask = attention_mask.float()
        fused = self.fusion(literal_feats, contextual_feats, mask)
        fused = self.drop(fused)
        return self.sent_head(fused), self.sarc_head(fused)

Writing idan_model.py


In [19]:
%%writefile idan_train.py
"""
idan_train.py — trains IDAN (Incongruity-Aware Dual-Attention Network).

Reuses the same infrastructure as transformer_multitask.py: source/variety
control tokens, stratified CV, early stopping, and the official ALTA metric.
Only the model architecture differs (see idan_model.py).
"""
from __future__ import annotations

import argparse
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from metrics import alta_score, competition_stratify_key
from transformer_multitask import SPECIAL_TOKENS, TextDataset
from idan_model import IDAN


@dataclass
class IDANConfig:
    backbone: str = "microsoft/deberta-v3-base"
    max_length: int = 192
    batch_size: int = 16
    epochs: int = 10
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.15
    cnn_out_channels: int = 256
    proj_dim: int = 384
    sentiment_loss_weight: float = 1.0
    sarcasm_loss_weight: float = 1.25
    gradient_accumulation_steps: int = 2
    seed: int = 42
    n_folds: int = 3
    early_stopping_patience: int = 2


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def build_model(cfg: IDANConfig, n_new_tokens: int):
    encoder = AutoModel.from_pretrained(cfg.backbone, torch_dtype=torch.float32)
    encoder.resize_token_embeddings(encoder.config.vocab_size + n_new_tokens)
    model = IDAN(
        encoder=encoder,
        hidden_size=encoder.config.hidden_size,
        cnn_out_channels=cfg.cnn_out_channels,
        proj_dim=cfg.proj_dim,
        dropout=cfg.dropout,
    )
    return model


def evaluate(model, loader, device):
    model.eval(); ps, pz = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            ps.append(torch.softmax(a, -1)[:, 1].cpu().numpy())
            pz.append(torch.softmax(b, -1)[:, 1].cpu().numpy())
    return np.concatenate(ps), np.concatenate(pz)


def train_one_fold(train_df, val_df, cfg, out_dir, fold):
    set_seed(cfg.seed + fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.backbone, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

    model = build_model(cfg, n_new_tokens=len(SPECIAL_TOKENS)).to(device)

    tr_ds = TextDataset(train_df, tokenizer, cfg.max_length, True)
    va_ds = TextDataset(val_df, tokenizer, cfg.max_length, True)
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_loader) * cfg.epochs // cfg.gradient_accumulation_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * cfg.warmup_ratio), steps)

    s_loss = nn.CrossEntropyLoss()
    z_loss = nn.CrossEntropyLoss()
    best, best_state, epochs_since_improve = -1.0, None, 0

    swa_state = None
    swa_count = 0
    SWA_TOLERANCE = 0.01

    for epoch in range(cfg.epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            loss = (cfg.sentiment_loss_weight * s_loss(a, batch["sentiment"].to(device))
                    + cfg.sarcasm_loss_weight * z_loss(b, batch["sarcasm"].to(device)))
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        scores, comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold} epoch={epoch+1} score={comp:.5f} {scores}")

        if comp > best:
            best = comp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if comp >= best - SWA_TOLERANCE:
            current_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state = current_state
                swa_count = 1
            else:
                swa_count += 1
                for k in swa_state:
                    if swa_state[k].dtype.is_floating_point:
                        swa_state[k] = swa_state[k] + (current_state[k] - swa_state[k]) / swa_count

        if epochs_since_improve >= cfg.early_stopping_patience:
            print(f"[IDAN] fold={fold}: no improvement for {cfg.early_stopping_patience} epochs, stopping early at epoch {epoch+1}")
            break

    if swa_state is not None and swa_count > 1:
        model.load_state_dict(swa_state)
        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        _, swa_comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold}: SWA (n={swa_count} epochs) score={swa_comp:.5f} vs best-single-epoch={best:.5f}")
        if swa_comp > best:
            print(f"[IDAN] fold={fold}: SWA wins, using averaged weights")
            best = swa_comp
            best_state = swa_state
        else:
            model.load_state_dict(best_state)

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), out_dir / f"fold{fold}.pt")
    with open(out_dir / f"fold{fold}.json", "w") as f:
        json.dump({"fold": fold, "best_validation_score": best, "config": asdict(cfg)}, f, indent=2)
    return model, tokenizer, device


def cv_train(train_csv, out_dir, cfg: IDANConfig, external_valid_csv=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(train_csv)
    keys = competition_stratify_key(df)
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)

    ext = pd.read_csv(external_valid_csv) if external_valid_csv else None
    ext_ps_all, ext_pz_all = [], []
    oof = df[["source", "variety", "text", "sentiment", "sarcasm"]].copy()
    oof["p_sentiment"] = np.nan; oof["p_sarcasm"] = np.nan

    for fold, (tr, va) in enumerate(skf.split(df, keys)):
        model, tokenizer, device = train_one_fold(df.iloc[tr].copy(), df.iloc[va].copy(), cfg, out, fold)

        va_ds = TextDataset(df.iloc[va].copy(), tokenizer, cfg.max_length, True)
        va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)
        ps, pz = evaluate(model, va_loader, device)
        oof.loc[df.index[va], "p_sentiment"] = ps
        oof.loc[df.index[va], "p_sarcasm"] = pz

        if ext is not None:
            ext_ds = TextDataset(ext.copy(), tokenizer, cfg.max_length, False)
            ext_loader = DataLoader(ext_ds, batch_size=cfg.batch_size * 2, shuffle=False)
            eps, epz = evaluate(model, ext_loader, device)
            ext_ps_all.append(eps); ext_pz_all.append(epz)

    oof.to_csv(out / "oof_probabilities.csv", index=False)
    pred = oof[["source", "variety", "text"]].copy()
    pred["sentiment"] = (oof.p_sentiment >= .5).astype(int)
    pred["sarcasm"] = (oof.p_sarcasm >= .5).astype(int)
    scores, comp = alta_score(df, pred)
    print("[IDAN] OOF @0.5", scores, comp)

    if ext is not None:
        ext_probs = ext[["source", "variety", "text"]].copy()
        ext_probs["p_sentiment"] = np.mean(ext_ps_all, axis=0)
        ext_probs["p_sarcasm"] = np.mean(ext_pz_all, axis=0)
        ext_probs.to_csv(out / "valid_probabilities.csv", index=False)
        ext_pred = ext_probs[["source", "variety", "text"]].copy()
        ext_pred["sentiment"] = (ext_probs.p_sentiment >= .5).astype(int)
        ext_pred["sarcasm"] = (ext_probs.p_sarcasm >= .5).astype(int)
        ext_scores, ext_comp = alta_score(ext, ext_pred)
        print("[IDAN] External valid @0.5", ext_scores, ext_comp)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="train.csv")
    ap.add_argument("--valid", default=None)
    ap.add_argument("--out", default="artifacts/idan")
    ap.add_argument("--backbone", default="microsoft/deberta-v3-base")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--max-length", type=int, default=192)
    ap.add_argument("--n-folds", type=int, default=3)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--grad-accum", type=int, default=2)
    args, _unknown = ap.parse_known_args()

    cfg = IDANConfig(
        backbone=args.backbone, epochs=args.epochs, batch_size=args.batch_size,
        max_length=args.max_length, n_folds=args.n_folds,
        early_stopping_patience=args.patience, gradient_accumulation_steps=args.grad_accum,
    )
    cv_train(args.train, args.out, cfg, args.valid)

Writing idan_train.py


In [20]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 198/198 [00:00<00:00, 896.78it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

In [21]:
from thresholds import optimize_thresholds, save_thresholds, apply_thresholds
from metrics import alta_score
import pandas as pd

valid = pd.read_csv('valid.csv')
probs = pd.read_csv('artifacts/idan/valid_probabilities.csv')
th = optimize_thresholds(valid, probs)
save_thresholds(th, 'artifacts/idan/thresholds.json')
pred = apply_thresholds(probs, th)
scores, final = alta_score(valid, pred)
print("IDAN CALIBRATED:", scores, final)

IDAN CALIBRATED: {'sentiment-en-AU': 0.921751020022255, 'sentiment-en-UK': 0.9557730284219963, 'sarcasm-en-AU': 0.7659760298898539, 'sarcasm-en-UK': 0.7358748778103616} 0.8288129489163083


In [22]:
%%writefile idan_model.py
"""
idan_model.py — Incongruity-Aware Dual-Attention Network (IDAN)
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class LiteralCNNBranch(nn.Module):
    def __init__(self, embed_dim: int, out_channels: int = 256, kernel_sizes=(2, 3, 4, 5)):
        super().__init__()
        per_k = out_channels // len(kernel_sizes)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, per_k, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.out_channels = per_k * len(kernel_sizes)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(self.out_channels)

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        x = token_embeds.transpose(1, 2)
        feats = []
        for conv in self.convs:
            f = self.act(conv(x))
            f = f[:, :, :token_embeds.size(1)]
            feats.append(f)
        out = torch.cat(feats, dim=1)
        out = out.transpose(1, 2)
        return self.norm(out)


class LiteralLinearBranch(nn.Module):
    def __init__(self, embed_dim: int, out_channels: int = 256):
        super().__init__()
        self.proj = nn.Linear(embed_dim, out_channels)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_channels)
        self.out_channels = out_channels

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        return self.norm(self.act(self.proj(token_embeds)))


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, dropout: float = 0.1):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, channels),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        avg_pool = (x * m).sum(dim=1) / denom
        max_pool = (x.masked_fill(m == 0, float("-inf"))).max(dim=1).values
        channel_att = torch.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool))
        return x * channel_att.unsqueeze(1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 5):
        super().__init__()
        self.conv = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        avg_pool = x.mean(dim=2, keepdim=True)
        max_pool = x.max(dim=2, keepdim=True).values
        pooled = torch.cat([avg_pool, max_pool], dim=2).transpose(1, 2)
        att = torch.sigmoid(self.conv(pooled)).transpose(1, 2)
        att = att * mask.unsqueeze(-1)
        return x * att


class IncongruityDualAttention(nn.Module):
    def __init__(self, literal_dim: int, contextual_dim: int, proj_dim: int = 384,
                 use_channel_attention: bool = True, use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True):
        super().__init__()
        self.lit_proj = nn.Linear(literal_dim, proj_dim)
        self.ctx_proj = nn.Linear(contextual_dim, proj_dim)
        joint_dim = proj_dim * 2
        self.use_channel_attention = use_channel_attention
        self.use_spatial_attention = use_spatial_attention
        self.use_incongruity_features = use_incongruity_features
        if use_channel_attention:
            self.channel_att = ChannelAttention(joint_dim)
        if use_spatial_attention:
            self.spatial_att = SpatialAttention()
        self.fuse_norm = nn.LayerNorm(joint_dim)
        self.proj_dim = proj_dim

    def output_dim(self) -> int:
        return self.proj_dim * 2 + (self.proj_dim * 2 if self.use_incongruity_features else 0)

    def forward(self, literal_feats, contextual_feats, mask):
        lit = self.lit_proj(literal_feats)
        ctx = self.ctx_proj(contextual_feats)
        joint = torch.cat([lit, ctx], dim=-1)

        if self.use_channel_attention:
            joint = self.channel_att(joint, mask)
        if self.use_spatial_attention:
            joint = self.spatial_att(joint, mask)
        joint = self.fuse_norm(joint)

        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        pooled_fused = (joint * m).sum(dim=1) / denom

        if not self.use_incongruity_features:
            return pooled_fused

        lit_pooled = (lit * m).sum(dim=1) / denom
        ctx_pooled = (ctx * m).sum(dim=1) / denom
        diff = lit_pooled - ctx_pooled
        prod = lit_pooled * ctx_pooled

        return torch.cat([pooled_fused, diff, prod], dim=-1)


class IDAN(nn.Module):
    def __init__(self, encoder, hidden_size: int, cnn_out_channels: int = 256,
                 proj_dim: int = 384, dropout: float = 0.15,
                 literal_encoder: str = "cnn",
                 use_channel_attention: bool = True,
                 use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True):
        super().__init__()
        self.encoder = encoder
        self.embed_layer = encoder.get_input_embeddings()

        if literal_encoder == "cnn":
            self.literal_branch = LiteralCNNBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        elif literal_encoder == "linear":
            self.literal_branch = LiteralLinearBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        else:
            raise ValueError(f"unknown literal_encoder: {literal_encoder}")

        self.fusion = IncongruityDualAttention(
            literal_dim=self.literal_branch.out_channels,
            contextual_dim=hidden_size,
            proj_dim=proj_dim,
            use_channel_attention=use_channel_attention,
            use_spatial_attention=use_spatial_attention,
            use_incongruity_features=use_incongruity_features,
        )
        fused_dim = self.fusion.output_dim()
        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))
        self.sarc_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        enc_out = self.encoder(**kwargs)
        contextual_feats = enc_out.last_hidden_state

        token_embeds = self.embed_layer(input_ids)
        literal_feats = self.literal_branch(token_embeds)

        mask = attention_mask.float()
        fused = self.fusion(literal_feats, contextual_feats, mask)
        fused = self.drop(fused)
        return self.sent_head(fused), self.sarc_head(fused)

Overwriting idan_model.py


In [23]:
%%writefile idan_train.py
"""
idan_train.py — trains IDAN (Incongruity-Aware Dual-Attention Network).
"""
from __future__ import annotations

import argparse
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from metrics import alta_score, competition_stratify_key
from transformer_multitask import SPECIAL_TOKENS, TextDataset
from idan_model import IDAN


@dataclass
class IDANConfig:
    backbone: str = "microsoft/deberta-v3-base"
    max_length: int = 192
    batch_size: int = 16
    epochs: int = 10
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.15
    cnn_out_channels: int = 256
    proj_dim: int = 384
    sentiment_loss_weight: float = 1.0
    sarcasm_loss_weight: float = 1.25
    gradient_accumulation_steps: int = 2
    seed: int = 42
    n_folds: int = 3
    early_stopping_patience: int = 2
    literal_encoder: str = "cnn"
    use_channel_attention: bool = True
    use_spatial_attention: bool = True
    use_incongruity_features: bool = True
    max_sarcasm_class_weight: float = 5.0


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def build_model(cfg: IDANConfig, n_new_tokens: int):
    encoder = AutoModel.from_pretrained(cfg.backbone, torch_dtype=torch.float32)
    encoder.resize_token_embeddings(encoder.config.vocab_size + n_new_tokens)
    model = IDAN(
        encoder=encoder,
        hidden_size=encoder.config.hidden_size,
        cnn_out_channels=cfg.cnn_out_channels,
        proj_dim=cfg.proj_dim,
        dropout=cfg.dropout,
        literal_encoder=cfg.literal_encoder,
        use_channel_attention=cfg.use_channel_attention,
        use_spatial_attention=cfg.use_spatial_attention,
        use_incongruity_features=cfg.use_incongruity_features,
    )
    return model


def evaluate(model, loader, device):
    model.eval(); ps, pz = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            ps.append(torch.softmax(a, -1)[:, 1].cpu().numpy())
            pz.append(torch.softmax(b, -1)[:, 1].cpu().numpy())
    return np.concatenate(ps), np.concatenate(pz)


def train_one_fold(train_df, val_df, cfg, out_dir, fold):
    set_seed(cfg.seed + fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.backbone, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

    model = build_model(cfg, n_new_tokens=len(SPECIAL_TOKENS)).to(device)

    tr_ds = TextDataset(train_df, tokenizer, cfg.max_length, True)
    va_ds = TextDataset(val_df, tokenizer, cfg.max_length, True)
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_loader) * cfg.epochs // cfg.gradient_accumulation_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * cfg.warmup_ratio), steps)

    s_loss = nn.CrossEntropyLoss()
    n_pos = max(int(train_df["sarcasm"].sum()), 1)
    n_neg = max(len(train_df) - n_pos, 1)
    pos_weight = min(n_neg / n_pos, cfg.max_sarcasm_class_weight)
    z_weight = torch.tensor([1.0, pos_weight], device=device)
    z_loss = nn.CrossEntropyLoss(weight=z_weight)
    print(f"[IDAN] fold={fold}: sarcasm class weight = {pos_weight:.2f} (n_pos={n_pos}, n_neg={n_neg})")
    best, best_state, epochs_since_improve = -1.0, None, 0

    swa_state = None
    swa_count = 0
    SWA_TOLERANCE = 0.01

    for epoch in range(cfg.epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            loss = (cfg.sentiment_loss_weight * s_loss(a, batch["sentiment"].to(device))
                    + cfg.sarcasm_loss_weight * z_loss(b, batch["sarcasm"].to(device)))
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        scores, comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold} epoch={epoch+1} score={comp:.5f} {scores}")

        if comp > best:
            best = comp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if comp >= best - SWA_TOLERANCE:
            current_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state = current_state
                swa_count = 1
            else:
                swa_count += 1
                for k in swa_state:
                    if swa_state[k].dtype.is_floating_point:
                        swa_state[k] = swa_state[k] + (current_state[k] - swa_state[k]) / swa_count

        if epochs_since_improve >= cfg.early_stopping_patience:
            print(f"[IDAN] fold={fold}: no improvement for {cfg.early_stopping_patience} epochs, stopping early at epoch {epoch+1}")
            break

    if swa_state is not None and swa_count > 1:
        model.load_state_dict(swa_state)
        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        _, swa_comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold}: SWA (n={swa_count} epochs) score={swa_comp:.5f} vs best-single-epoch={best:.5f}")
        if swa_comp > best:
            print(f"[IDAN] fold={fold}: SWA wins, using averaged weights")
            best = swa_comp
            best_state = swa_state
        else:
            model.load_state_dict(best_state)

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), out_dir / f"fold{fold}.pt")
    with open(out_dir / f"fold{fold}.json", "w") as f:
        json.dump({"fold": fold, "best_validation_score": best, "config": asdict(cfg)}, f, indent=2)
    return model, tokenizer, device


def cv_train(train_csv, out_dir, cfg: IDANConfig, external_valid_csv=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(train_csv)
    keys = competition_stratify_key(df)
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)

    ext = pd.read_csv(external_valid_csv) if external_valid_csv else None
    ext_ps_all, ext_pz_all = [], []
    oof = df[["source", "variety", "text", "sentiment", "sarcasm"]].copy()
    oof["p_sentiment"] = np.nan; oof["p_sarcasm"] = np.nan

    for fold, (tr, va) in enumerate(skf.split(df, keys)):
        model, tokenizer, device = train_one_fold(df.iloc[tr].copy(), df.iloc[va].copy(), cfg, out, fold)

        va_ds = TextDataset(df.iloc[va].copy(), tokenizer, cfg.max_length, True)
        va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)
        ps, pz = evaluate(model, va_loader, device)
        oof.loc[df.index[va], "p_sentiment"] = ps
        oof.loc[df.index[va], "p_sarcasm"] = pz

        if ext is not None:
            ext_ds = TextDataset(ext.copy(), tokenizer, cfg.max_length, False)
            ext_loader = DataLoader(ext_ds, batch_size=cfg.batch_size * 2, shuffle=False)
            eps, epz = evaluate(model, ext_loader, device)
            ext_ps_all.append(eps); ext_pz_all.append(epz)

    oof.to_csv(out / "oof_probabilities.csv", index=False)
    pred = oof[["source", "variety", "text"]].copy()
    pred["sentiment"] = (oof.p_sentiment >= .5).astype(int)
    pred["sarcasm"] = (oof.p_sarcasm >= .5).astype(int)
    scores, comp = alta_score(df, pred)
    print("[IDAN] OOF @0.5", scores, comp)

    if ext is not None:
        ext_probs = ext[["source", "variety", "text"]].copy()
        ext_probs["p_sentiment"] = np.mean(ext_ps_all, axis=0)
        ext_probs["p_sarcasm"] = np.mean(ext_pz_all, axis=0)
        ext_probs.to_csv(out / "valid_probabilities.csv", index=False)
        ext_pred = ext_probs[["source", "variety", "text"]].copy()
        ext_pred["sentiment"] = (ext_probs.p_sentiment >= .5).astype(int)
        ext_pred["sarcasm"] = (ext_probs.p_sarcasm >= .5).astype(int)
        ext_scores, ext_comp = alta_score(ext, ext_pred)
        print("[IDAN] External valid @0.5", ext_scores, ext_comp)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="train.csv")
    ap.add_argument("--valid", default=None)
    ap.add_argument("--out", default="artifacts/idan")
    ap.add_argument("--backbone", default="microsoft/deberta-v3-base")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--max-length", type=int, default=192)
    ap.add_argument("--n-folds", type=int, default=3)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--grad-accum", type=int, default=2)
    ap.add_argument("--literal-encoder", choices=["cnn", "linear"], default="cnn")
    ap.add_argument("--no-channel-attention", action="store_true")
    ap.add_argument("--no-spatial-attention", action="store_true")
    ap.add_argument("--no-incongruity-features", action="store_true")
    args, _unknown = ap.parse_known_args()

    cfg = IDANConfig(
        backbone=args.backbone, epochs=args.epochs, batch_size=args.batch_size,
        max_length=args.max_length, n_folds=args.n_folds,
        early_stopping_patience=args.patience, gradient_accumulation_steps=args.grad_accum,
        literal_encoder=args.literal_encoder,
        use_channel_attention=not args.no_channel_attention,
        use_spatial_attention=not args.no_spatial_attention,
        use_incongruity_features=not args.no_incongruity_features,
    )
    cv_train(args.train, args.out, cfg, args.valid)

Overwriting idan_train.py


In [24]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan_v2 \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 198/198 [00:00<00:00, 1010.55it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

In [25]:
!pip install -q -U transformers accelerate sentencepiece safetensors
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [26]:
!cp -r /kaggle/input/datasets/rifatbinreza/alta2026-pipeline/alta2026_pipeline /kaggle/working/
%cd /kaggle/working/alta2026_pipeline
!ls

/kaggle/working/alta2026_pipeline
all_artifacts.zip	  evaluate.py	 requirements.txt
answer.csv		  idan_model.py  run_baseline.sh
answer_initial.csv	  idan_train.py  sweep_blend.py
answer.zip		  infer.py	 thresholds.py
artifacts		  metadata	 train.csv
blend.py		  metrics.py	 transformer_multitask.py
classical_baseline.py	  __pycache__	 validate_answer.py
classical_baseline_v2.py  README.md	 valid.csv


In [27]:
%%writefile idan_model.py
"""
idan_model.py — Incongruity-Aware Dual-Attention Network (IDAN)
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class LiteralCNNBranch(nn.Module):
    def __init__(self, embed_dim: int, out_channels: int = 256, kernel_sizes=(2, 3, 4, 5)):
        super().__init__()
        per_k = out_channels // len(kernel_sizes)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, per_k, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.out_channels = per_k * len(kernel_sizes)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(self.out_channels)

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        x = token_embeds.transpose(1, 2)
        feats = []
        for conv in self.convs:
            f = self.act(conv(x))
            f = f[:, :, :token_embeds.size(1)]
            feats.append(f)
        out = torch.cat(feats, dim=1)
        out = out.transpose(1, 2)
        return self.norm(out)


class LiteralLinearBranch(nn.Module):
    def __init__(self, embed_dim: int, out_channels: int = 256):
        super().__init__()
        self.proj = nn.Linear(embed_dim, out_channels)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_channels)
        self.out_channels = out_channels

    def forward(self, token_embeds: torch.Tensor) -> torch.Tensor:
        return self.norm(self.act(self.proj(token_embeds)))


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, dropout: float = 0.1):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, channels),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        avg_pool = (x * m).sum(dim=1) / denom
        max_pool = (x.masked_fill(m == 0, float("-inf"))).max(dim=1).values
        channel_att = torch.sigmoid(self.mlp(avg_pool) + self.mlp(max_pool))
        return x * channel_att.unsqueeze(1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 5):
        super().__init__()
        self.conv = nn.Conv1d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        avg_pool = x.mean(dim=2, keepdim=True)
        max_pool = x.max(dim=2, keepdim=True).values
        pooled = torch.cat([avg_pool, max_pool], dim=2).transpose(1, 2)
        att = torch.sigmoid(self.conv(pooled)).transpose(1, 2)
        att = att * mask.unsqueeze(-1)
        return x * att


class IncongruityDualAttention(nn.Module):
    def __init__(self, literal_dim: int, contextual_dim: int, proj_dim: int = 384,
                 use_channel_attention: bool = True, use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True):
        super().__init__()
        self.lit_proj = nn.Linear(literal_dim, proj_dim)
        self.ctx_proj = nn.Linear(contextual_dim, proj_dim)
        joint_dim = proj_dim * 2
        self.use_channel_attention = use_channel_attention
        self.use_spatial_attention = use_spatial_attention
        self.use_incongruity_features = use_incongruity_features
        if use_channel_attention:
            self.channel_att = ChannelAttention(joint_dim)
        if use_spatial_attention:
            self.spatial_att = SpatialAttention()
        self.fuse_norm = nn.LayerNorm(joint_dim)
        self.proj_dim = proj_dim

    def output_dim(self) -> int:
        return self.proj_dim * 2 + (self.proj_dim * 2 if self.use_incongruity_features else 0)

    def forward(self, literal_feats, contextual_feats, mask):
        lit = self.lit_proj(literal_feats)
        ctx = self.ctx_proj(contextual_feats)
        joint = torch.cat([lit, ctx], dim=-1)

        if self.use_channel_attention:
            joint = self.channel_att(joint, mask)
        if self.use_spatial_attention:
            joint = self.spatial_att(joint, mask)
        joint = self.fuse_norm(joint)

        m = mask.unsqueeze(-1)
        denom = m.sum(dim=1).clamp(min=1.0)
        pooled_fused = (joint * m).sum(dim=1) / denom

        if not self.use_incongruity_features:
            return pooled_fused

        lit_pooled = (lit * m).sum(dim=1) / denom
        ctx_pooled = (ctx * m).sum(dim=1) / denom
        diff = lit_pooled - ctx_pooled
        prod = lit_pooled * ctx_pooled

        return torch.cat([pooled_fused, diff, prod], dim=-1)


class IDAN(nn.Module):
    def __init__(self, encoder, hidden_size: int, cnn_out_channels: int = 256,
                 proj_dim: int = 384, dropout: float = 0.15,
                 literal_encoder: str = "cnn",
                 use_channel_attention: bool = True,
                 use_spatial_attention: bool = True,
                 use_incongruity_features: bool = True):
        super().__init__()
        self.encoder = encoder
        self.embed_layer = encoder.get_input_embeddings()

        if literal_encoder == "cnn":
            self.literal_branch = LiteralCNNBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        elif literal_encoder == "linear":
            self.literal_branch = LiteralLinearBranch(
                embed_dim=self.embed_layer.embedding_dim, out_channels=cnn_out_channels
            )
        else:
            raise ValueError(f"unknown literal_encoder: {literal_encoder}")

        self.fusion = IncongruityDualAttention(
            literal_dim=self.literal_branch.out_channels,
            contextual_dim=hidden_size,
            proj_dim=proj_dim,
            use_channel_attention=use_channel_attention,
            use_spatial_attention=use_spatial_attention,
            use_incongruity_features=use_incongruity_features,
        )
        fused_dim = self.fusion.output_dim()
        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))
        self.sarc_head = nn.Sequential(nn.Linear(fused_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, 2))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        enc_out = self.encoder(**kwargs)
        contextual_feats = enc_out.last_hidden_state

        token_embeds = self.embed_layer(input_ids)
        literal_feats = self.literal_branch(token_embeds)

        mask = attention_mask.float()
        fused = self.fusion(literal_feats, contextual_feats, mask)
        fused = self.drop(fused)
        return self.sent_head(fused), self.sarc_head(fused)

Overwriting idan_model.py


In [28]:
%%writefile idan_train.py
"""
idan_train.py — trains IDAN (Incongruity-Aware Dual-Attention Network).
"""
from __future__ import annotations

import argparse
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from metrics import alta_score, competition_stratify_key
from transformer_multitask import SPECIAL_TOKENS, TextDataset
from idan_model import IDAN


@dataclass
class IDANConfig:
    backbone: str = "microsoft/deberta-v3-base"
    max_length: int = 192
    batch_size: int = 16
    epochs: int = 10
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.15
    cnn_out_channels: int = 256
    proj_dim: int = 384
    sentiment_loss_weight: float = 1.0
    sarcasm_loss_weight: float = 1.25
    gradient_accumulation_steps: int = 2
    seed: int = 42
    n_folds: int = 3
    early_stopping_patience: int = 2
    literal_encoder: str = "cnn"
    use_channel_attention: bool = True
    use_spatial_attention: bool = True
    use_incongruity_features: bool = True
    max_sarcasm_class_weight: float = 5.0


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def build_model(cfg: IDANConfig, n_new_tokens: int):
    encoder = AutoModel.from_pretrained(cfg.backbone, torch_dtype=torch.float32)
    encoder.resize_token_embeddings(encoder.config.vocab_size + n_new_tokens)
    model = IDAN(
        encoder=encoder,
        hidden_size=encoder.config.hidden_size,
        cnn_out_channels=cfg.cnn_out_channels,
        proj_dim=cfg.proj_dim,
        dropout=cfg.dropout,
        literal_encoder=cfg.literal_encoder,
        use_channel_attention=cfg.use_channel_attention,
        use_spatial_attention=cfg.use_spatial_attention,
        use_incongruity_features=cfg.use_incongruity_features,
    )
    return model


def evaluate(model, loader, device):
    model.eval(); ps, pz = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            ps.append(torch.softmax(a, -1)[:, 1].cpu().numpy())
            pz.append(torch.softmax(b, -1)[:, 1].cpu().numpy())
    return np.concatenate(ps), np.concatenate(pz)


def train_one_fold(train_df, val_df, cfg, out_dir, fold):
    set_seed(cfg.seed + fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.backbone, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

    model = build_model(cfg, n_new_tokens=len(SPECIAL_TOKENS)).to(device)

    tr_ds = TextDataset(train_df, tokenizer, cfg.max_length, True)
    va_ds = TextDataset(val_df, tokenizer, cfg.max_length, True)
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_loader) * cfg.epochs // cfg.gradient_accumulation_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps * cfg.warmup_ratio), steps)

    s_loss = nn.CrossEntropyLoss()
    n_pos = max(int(train_df["sarcasm"].sum()), 1)
    n_neg = max(len(train_df) - n_pos, 1)
    pos_weight = min(n_neg / n_pos, cfg.max_sarcasm_class_weight)
    z_weight = torch.tensor([1.0, pos_weight], device=device)
    z_loss = nn.CrossEntropyLoss(weight=z_weight)
    print(f"[IDAN] fold={fold}: sarcasm class weight = {pos_weight:.2f} (n_pos={n_pos}, n_neg={n_neg})")
    best, best_state, epochs_since_improve = -1.0, None, 0

    swa_state = None
    swa_count = 0
    SWA_TOLERANCE = 0.01

    for epoch in range(cfg.epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            loss = (cfg.sentiment_loss_weight * s_loss(a, batch["sentiment"].to(device))
                    + cfg.sarcasm_loss_weight * z_loss(b, batch["sarcasm"].to(device)))
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        scores, comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold} epoch={epoch+1} score={comp:.5f} {scores}")

        if comp > best:
            best = comp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if comp >= best - SWA_TOLERANCE:
            current_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state = current_state
                swa_count = 1
            else:
                swa_count += 1
                for k in swa_state:
                    if swa_state[k].dtype.is_floating_point:
                        swa_state[k] = swa_state[k] + (current_state[k] - swa_state[k]) / swa_count

        if epochs_since_improve >= cfg.early_stopping_patience:
            print(f"[IDAN] fold={fold}: no improvement for {cfg.early_stopping_patience} epochs, stopping early at epoch {epoch+1}")
            break

    if swa_state is not None and swa_count > 1:
        model.load_state_dict(swa_state)
        ps, pz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        _, swa_comp = alta_score(val_df, tmp)
        print(f"[IDAN] fold={fold}: SWA (n={swa_count} epochs) score={swa_comp:.5f} vs best-single-epoch={best:.5f}")
        if swa_comp > best:
            print(f"[IDAN] fold={fold}: SWA wins, using averaged weights")
            best = swa_comp
            best_state = swa_state
        else:
            model.load_state_dict(best_state)

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), out_dir / f"fold{fold}.pt")
    with open(out_dir / f"fold{fold}.json", "w") as f:
        json.dump({"fold": fold, "best_validation_score": best, "config": asdict(cfg)}, f, indent=2)
    return model, tokenizer, device


def cv_train(train_csv, out_dir, cfg: IDANConfig, external_valid_csv=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(train_csv)
    keys = competition_stratify_key(df)
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)

    ext = pd.read_csv(external_valid_csv) if external_valid_csv else None
    ext_ps_all, ext_pz_all = [], []
    oof = df[["source", "variety", "text", "sentiment", "sarcasm"]].copy()
    oof["p_sentiment"] = np.nan; oof["p_sarcasm"] = np.nan

    for fold, (tr, va) in enumerate(skf.split(df, keys)):
        model, tokenizer, device = train_one_fold(df.iloc[tr].copy(), df.iloc[va].copy(), cfg, out, fold)

        va_ds = TextDataset(df.iloc[va].copy(), tokenizer, cfg.max_length, True)
        va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)
        ps, pz = evaluate(model, va_loader, device)
        oof.loc[df.index[va], "p_sentiment"] = ps
        oof.loc[df.index[va], "p_sarcasm"] = pz

        if ext is not None:
            ext_ds = TextDataset(ext.copy(), tokenizer, cfg.max_length, False)
            ext_loader = DataLoader(ext_ds, batch_size=cfg.batch_size * 2, shuffle=False)
            eps, epz = evaluate(model, ext_loader, device)
            ext_ps_all.append(eps); ext_pz_all.append(epz)

    oof.to_csv(out / "oof_probabilities.csv", index=False)
    pred = oof[["source", "variety", "text"]].copy()
    pred["sentiment"] = (oof.p_sentiment >= .5).astype(int)
    pred["sarcasm"] = (oof.p_sarcasm >= .5).astype(int)
    scores, comp = alta_score(df, pred)
    print("[IDAN] OOF @0.5", scores, comp)

    if ext is not None:
        ext_probs = ext[["source", "variety", "text"]].copy()
        ext_probs["p_sentiment"] = np.mean(ext_ps_all, axis=0)
        ext_probs["p_sarcasm"] = np.mean(ext_pz_all, axis=0)
        ext_probs.to_csv(out / "valid_probabilities.csv", index=False)
        ext_pred = ext_probs[["source", "variety", "text"]].copy()
        ext_pred["sentiment"] = (ext_probs.p_sentiment >= .5).astype(int)
        ext_pred["sarcasm"] = (ext_probs.p_sarcasm >= .5).astype(int)
        ext_scores, ext_comp = alta_score(ext, ext_pred)
        print("[IDAN] External valid @0.5", ext_scores, ext_comp)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="train.csv")
    ap.add_argument("--valid", default=None)
    ap.add_argument("--out", default="artifacts/idan")
    ap.add_argument("--backbone", default="microsoft/deberta-v3-base")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--max-length", type=int, default=192)
    ap.add_argument("--n-folds", type=int, default=3)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--grad-accum", type=int, default=2)
    ap.add_argument("--literal-encoder", choices=["cnn", "linear"], default="cnn")
    ap.add_argument("--no-channel-attention", action="store_true")
    ap.add_argument("--no-spatial-attention", action="store_true")
    ap.add_argument("--no-incongruity-features", action="store_true")
    args, _unknown = ap.parse_known_args()

    cfg = IDANConfig(
        backbone=args.backbone, epochs=args.epochs, batch_size=args.batch_size,
        max_length=args.max_length, n_folds=args.n_folds,
        early_stopping_patience=args.patience, gradient_accumulation_steps=args.grad_accum,
        literal_encoder=args.literal_encoder,
        use_channel_attention=not args.no_channel_attention,
        use_spatial_attention=not args.no_spatial_attention,
        use_incongruity_features=not args.no_incongruity_features,
    )
    cv_train(args.train, args.out, cfg, args.valid)

Overwriting idan_train.py


In [29]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan_v2 \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 198/198 [00:00<00:00, 1113.49it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

In [30]:
from thresholds import optimize_thresholds, save_thresholds, apply_thresholds
from metrics import alta_score
import pandas as pd
valid = pd.read_csv('valid.csv')
probs = pd.read_csv('artifacts/idan_v2/valid_probabilities.csv')
th = optimize_thresholds(valid, probs)
save_thresholds(th, 'artifacts/idan_v2/thresholds.json')
pred = apply_thresholds(probs, th)
scores, final = alta_score(valid, pred)
print('IDAN v2 (class-weighted) CALIBRATED:', scores, final)

IDAN v2 (class-weighted) CALIBRATED: {'sentiment-en-AU': 0.9211708246327435, 'sentiment-en-UK': 0.955801625950548, 'sarcasm-en-AU': 0.7963915757700487, 'sarcasm-en-UK': 0.7018627896410722} 0.8115168071369079


In [31]:
!python idan_train.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/idan_noincong \
  --backbone microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 16 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 2 \
  --no-incongruity-features

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████| 198/198 [00:00<00:00, 1106.30it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

In [32]:
probs2 = pd.read_csv('artifacts/idan_noincong/valid_probabilities.csv')
th2 = optimize_thresholds(valid, probs2)
save_thresholds(th2, 'artifacts/idan_noincong/thresholds.json')
pred2 = apply_thresholds(probs2, th2)
scores2, final2 = alta_score(valid, pred2)
print('NO-INCONGRUITY ABLATION CALIBRATED:', scores2, final2)

NO-INCONGRUITY ABLATION CALIBRATED: {'sentiment-en-AU': 0.9155561429400061, 'sentiment-en-UK': 0.9584141348847232, 'sarcasm-en-AU': 0.8002647720753717, 'sarcasm-en-UK': 0.7285567900986518} 0.822056466519329


In [33]:
!cd /kaggle/working/alta2026_pipeline && zip -r idan_run_artifacts.zip artifacts/idan_v2 artifacts/idan_noincong

  adding: artifacts/idan_v2/ (stored 0%)
  adding: artifacts/idan_v2/oof_probabilities.csv (deflated 59%)
  adding: artifacts/idan_v2/fold1.json (deflated 49%)
  adding: artifacts/idan_v2/fold0.pt (deflated 21%)
  adding: artifacts/idan_v2/thresholds.json (deflated 41%)
  adding: artifacts/idan_v2/fold0.json (deflated 49%)
  adding: artifacts/idan_v2/valid_probabilities.csv (deflated 59%)
  adding: artifacts/idan_v2/fold2.pt
zip I/O error: No space left on device
zip error: Output file write failure (write error on zip file)
